# HW#5 -- Part II: Summarization by Fine-Tuning Encoder-Decoder Models

**Name:** [Your Name]
**Course:** CSC 583 -- Natural Language Processing, Fall 2025
**Assignment:** HW#5, Part II
**Collaborators:** [List any collaborators, or "None"]

## 0. Setup (Google Colab)

This part is done for you. We mount Drive purely for **output persistence** -- so the
fine-tuned checkpoint and the CSV results survive a runtime disconnect. Input never touches
Drive: the SAMSum dataset and both model checkpoints (original + GPT-2 for perplexity) are
pulled fresh from the HuggingFace Hub.

In [ ]:
# Mounted only so the checkpoint + CSV outputs (Section 7 onward) persist across sessions --
# the SAMSum dataset and the pretrained models are pulled fresh from the HF Hub over the
# network, so Drive is never needed for input.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

thisdir = "/content/drive/My Drive/CSC583_Fall2026/HW5"   # <-- update to YOUR folder if different
os.chdir(thisdir)
!pwd

In [ ]:
!pip install -q -U transformers tokenizers accelerate huggingface_hub datasets evaluate sentencepiece rouge_score bert_score

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from evaluate import load as load_metric

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Load the SAMSum dataset

Load `knkarthick/samsum` from the HF hub. It already comes split into train/validation/test
-- use those splits as-is. Use `'dialogue'` as the original text and `'summary'` as the
reference summary.

In [ ]:
# TODO (Section 1): Load the SAMSum dataset.
#
# Required variable when you're done:
#   raw_ds -- the DatasetDict returned by load_dataset(...), with "train"/"validation"/"test" splits

raw_ds = None   # TODO

raise NotImplementedError("TODO: load the SAMSum dataset (Section 1)")

## 2. Load model + tokenizer

Fine-tune `google-t5/t5-small` (or `google/flan-t5-small` as a drop-in alternative). Load it
now, *before* touching the training data -- Section 4 below evaluates this exact off-the-shelf
checkpoint as your baseline, prior to any fine-tuning.

In [ ]:
# TODO (Section 2): Load the model + tokenizer you'll fine-tune.
#
# Required variables when you're done:
#   model_name -- the HF model id you chose
#   tokenizer  -- AutoTokenizer instance
#   model      -- AutoModelForSeq2SeqLM instance, moved to device

model_name = None   # TODO

raise NotImplementedError("TODO: load the model + tokenizer (Section 2)")

## 3. Build the test-set evaluation pipeline (ROUGE + Perplexity + BERTScore)

Build this now, before training, so you can run the exact same pipeline once on the untouched,
off-the-shelf model (Section 4) and again later on the fine-tuned model (Section 8). The test
set must be preprocessed with `padding=True` + `.set_format("torch")`, exactly as in Part I --
this is a separate, model-agnostic preprocessing from the `labels`-based tokenization you'll
use for training in Section 5.

**Perplexity note:** see the assignment's "KNOWN ISSUE" under Part I's Perplexity example --
the `evaluate` "perplexity" metric script is currently broken on Colab's default package
versions. You'll likely need the same GPT-2-based workaround you used in Part I.

In [ ]:
# TODO (Section 3): Build the pieces you'll need for a full test-set evaluation.
#
# Required variables/functions when you're done:
#   prepare_for_inference(dataset, tokenizer, text_column, ...) -> tokenized Dataset (torch format)
#   generate_summaries(model, tokenizer, tokenized_ds, ...)     -> list[str] predictions
#   evaluate_summaries(predictions, references)                -> dict of the 6 metric values
#     (rouge1, rouge2, mean_perplexity, bertscore_precision, bertscore_recall, bertscore_f1)
#   rouge            -- load_metric("rouge")
#   bertscore        -- load_metric("bertscore")
#   test_ds_raw      -- raw_ds["test"]
#   test_tokenized   -- test_ds_raw prepared with prepare_for_inference
#   references       -- test_ds_raw["summary"]

raise NotImplementedError("TODO: build the test-set evaluation pipeline (Section 3)")

## 4. Baseline: run and evaluate the ORIGINAL (off-the-shelf) model, BEFORE fine-tuning

This is the same experiment as Part I -- a pure inference task, `model.generate()` on the
untouched pretrained checkpoint. Do this *first*, before any training happens, so your baseline
numbers are guaranteed to reflect the off-the-shelf model and nothing else.

In [ ]:
# TODO (Section 4): Evaluate the untouched, off-the-shelf `model` on the test set.
#
# Required variables when you're done:
#   original_predictions -- list[str], from generate_summaries(model, tokenizer, test_tokenized)
#   original_metrics      -- dict, from evaluate_summaries(original_predictions, references)

raise NotImplementedError("TODO: evaluate the original off-the-shelf model (Section 4)")

## 5. Preprocess the training data into a tokenized `DatasetDict`

Use the same `"summarize: "` prefix convention as Part I. Tokenize the `'dialogue'` column as
input and the `'summary'` column as the target (the modern HF way is
`tokenizer(text_target=...)` to build `labels`).

In [ ]:
# TODO (Section 5): Write preprocess_function and map it over raw_ds.
#
# Required variable when you're done:
#   tokenized_ds -- a DatasetDict with "input_ids", "attention_mask", "labels" columns
#                   for each of train/validation/test

prefix = "summarize: "

def preprocess_function(examples):
    raise NotImplementedError("TODO: fill in preprocess_function (Section 5)")

tokenized_ds = None   # TODO: raw_ds.map(preprocess_function, batched=True, remove_columns=...)

raise NotImplementedError("TODO: preprocess the training data (Section 5)")

## 6. Set up the evaluator used *during training* (on the validation set)

Report ROUGE at each epoch during training, using `Seq2SeqTrainer`'s `compute_metrics`
callback. (Remember: predictions/labels here are token IDs -- you'll need to decode them,
and replace `-100` label positions with the pad token id before decoding. Reuse the `rouge`
metric you already loaded in Section 3.)

In [ ]:
# TODO (Section 6): Write compute_metrics(eval_preds), using the `rouge` metric from Section 3.
#
# Required variable when you're done:
#   compute_metrics -- a function(eval_preds) -> dict, suitable for Seq2SeqTrainer

def compute_metrics(eval_preds):
    raise NotImplementedError("TODO: fill in compute_metrics (Section 6)")

## 7. Train the model (3 epochs)

Use `Seq2SeqTrainingArguments` + `Seq2SeqTrainer`. Remember: the results reported during
training are with respect to the **validation** set, not train.

This will train `model` in place -- the same object you already evaluated as the untouched
baseline in Section 4, so there's no ambiguity about which checkpoint produced which numbers.

In [ ]:
# TODO (Section 7a): Build the data collator, training args, and trainer.
#
# Required variable when you're done:
#   trainer -- a configured Seq2SeqTrainer, not yet trained

data_collator = None      # TODO
training_args = None      # TODO: Seq2SeqTrainingArguments(...)
trainer = None             # TODO: Seq2SeqTrainer(...)

raise NotImplementedError("TODO: configure the trainer (Section 7a)")

In [ ]:
# TODO (Section 7b): Train, and save the fine-tuned model to disk.
#
# trainer.train()
# trainer.save_model("t5-small-samsum-finetuned")

raise NotImplementedError("TODO: train and save the model (Section 7b)")

## 8. Evaluate the FINE-TUNED model on the TEST set

Reload the saved checkpoint fresh from disk (rather than reusing the in-memory `model`) so this
evaluation is unambiguously the one saved by the trainer, and reuse the exact same
`generate_summaries` / `evaluate_summaries` pipeline from Section 3.

In [ ]:
# TODO (Section 8): Evaluate your FINE-TUNED model on the test set.
#
# Required variables when you're done:
#   finetuned_model       -- AutoModelForSeq2SeqLM.from_pretrained("t5-small-samsum-finetuned")
#   finetuned_predictions -- list[str], from generate_summaries(...)
#   finetuned_metrics     -- dict, from evaluate_summaries(...)

raise NotImplementedError("TODO: evaluate the fine-tuned model (Section 8)")

## 9. Compare the ORIGINAL (baseline) model with the fine-tuned model

Both sets of metrics were computed with the identical pipeline on the identical test set --
`original_metrics` from Section 4 (before fine-tuning), `finetuned_metrics` from Section 8
(after fine-tuning).

In [ ]:
# TODO (Section 9): Build a comparison table from the two metrics dicts you already have.
#
# Required variable when you're done:
#   comparison -- a DataFrame with columns "original (off-the-shelf)" and "fine-tuned"

raise NotImplementedError("TODO: build the comparison table (Section 9)")

## 10. Two qualitative examples: ground truth vs. generated (before/after fine-tuning)

**Note:** reuse these same two test-set indices in Part III, so you can compare the
encoder-decoder outputs directly against the SOTA-LLM-prompted outputs on identical inputs.

In [ ]:
example_idx = [10, 50]   # <-- reuse these same indices in Part III (or pick your own,
                          #     just keep them consistent across Part II and Part III)

# TODO (Section 10): For each index in example_idx, print the dialogue, reference summary,
# original model's summary, and fine-tuned model's summary.

raise NotImplementedError("TODO: print the qualitative comparison (Section 10)")

## 11. Save results (for the write-up)

In [ ]:
# TODO (Section 11): Save comparison and your two qualitative examples to CSV.

raise NotImplementedError("TODO: save your results (Section 11)")